# 📊 Trader Performance vs Market Sentiment
### Primetrade.ai — Data Science Intern Assignment

**Objective:** Analyze how Bitcoin Fear/Greed sentiment relates to trader behavior and performance on Hyperliquid DEX.

**Datasets:**
- `fear_greed_index.csv` — Daily Bitcoin Fear/Greed classification (2018–2025)
- `historical_data.csv` — Historical Hyperliquid trader data

**Confirmed columns in trades data:**
`Account, Coin, Execution Price, Size Tokens, Size USD, Side, Timestamp IST, Start Position, Direction, Closed PnL, Transaction Hash, Order ID, Crossed, Fee, Trade ID, Timestamp`

---

## 🗂️ PART A — Data Preparation

In [ ]:
# ── Cell 1: Import Libraries ──────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size'] = 10

print('✅ All libraries loaded successfully')

In [ ]:
# ── Cell 2: Load Both Datasets ────────────────────────────────────────────────
# NOTE: Place both CSV files in the SAME folder as this notebook before running

sentiment_df = pd.read_csv('fear_greed_index.csv')
trades_df    = pd.read_csv('historical_data.csv')   # rename your file to this

print('━'*55)
print('📁 DATASET 1 — Fear/Greed Sentiment Index')
print(f'   Rows    : {sentiment_df.shape[0]}')
print(f'   Columns : {sentiment_df.shape[1]}')
print(f'   Names   : {sentiment_df.columns.tolist()}')

print()
print('📁 DATASET 2 — Hyperliquid Trades')
print(f'   Rows    : {trades_df.shape[0]}')
print(f'   Columns : {trades_df.shape[1]}')
print(f'   Names   : {trades_df.columns.tolist()}')
print('━'*55)

print('\n--- Sentiment (first 3 rows) ---')
display(sentiment_df.head(3))
print('\n--- Trades (first 3 rows) ---')
display(trades_df.head(3))

In [ ]:
# ── Cell 3: Missing Values & Duplicates ───────────────────────────────────────
print('=== MISSING VALUES — SENTIMENT ===')
print(sentiment_df.isnull().sum().to_string())

print('\n=== MISSING VALUES — TRADES ===')
print(trades_df.isnull().sum().to_string())

print(f'\nSentiment duplicates : {sentiment_df.duplicated().sum()}')
print(f'Trades duplicates    : {trades_df.duplicated().sum()}')

# Remove duplicates
sentiment_df.drop_duplicates(inplace=True)
trades_df.drop_duplicates(inplace=True)

# Fill missing PnL / Size with 0 (open trades not yet closed)
trades_df['Closed PnL'] = trades_df['Closed PnL'].fillna(0)
trades_df['Size USD']   = trades_df['Size USD'].fillna(0)

print('\n✅ Cleaning done — duplicates removed, missing numerics filled with 0')

In [ ]:
# ── Cell 4: Convert Timestamps & Align Dates ──────────────────────────────────
# 'Timestamp IST' format is  DD-MM-YYYY HH:MM  e.g. '02-12-2024 22:50'
trades_df['date'] = pd.to_datetime(
    trades_df['Timestamp IST'], format='%d-%m-%Y %H:%M'
).dt.normalize()    # strip time → keep date at midnight

# Sentiment 'date' is already YYYY-MM-DD
sentiment_df['date'] = pd.to_datetime(sentiment_df['date'])

print('Trades date range   :', trades_df['date'].min().date(), '→', trades_df['date'].max().date())
print('Sentiment date range:', sentiment_df['date'].min().date(), '→', sentiment_df['date'].max().date())

In [ ]:
# ── Cell 5: Merge Both Datasets on Date ───────────────────────────────────────
merged_df = trades_df.merge(
    sentiment_df[['date', 'classification', 'value']],
    on='date',
    how='left'
)
merged_df.rename(columns={
    'classification': 'sentiment',
    'value'         : 'fear_greed_score'
}, inplace=True)

unmatched = merged_df['sentiment'].isnull().sum()
print(f'Rows with no sentiment match : {unmatched} ({unmatched/len(merged_df)*100:.1f}%)')
merged_df.dropna(subset=['sentiment'], inplace=True)

# Collapse 5 classes → 3 for cleaner charts
sentiment_map = {
    'Extreme Fear' : 'Fear',
    'Fear'         : 'Fear',
    'Neutral'      : 'Neutral',
    'Greed'        : 'Greed',
    'Extreme Greed': 'Greed'
}
merged_df['sentiment_broad'] = merged_df['sentiment'].map(sentiment_map)

print(f'Final merged rows: {len(merged_df)}')
print('\nSentiment (5-class):')
print(merged_df['sentiment'].value_counts().to_string())
print('\nSentiment (3-class broad):')
print(merged_df['sentiment_broad'].value_counts().to_string())

In [ ]:
# ── Cell 6: Engineer Key Metrics (Daily per Trader) ───────────────────────────

# Trade-level flags
merged_df['is_long'] = merged_df['Side'].str.upper().isin(['BUY', 'LONG'])
merged_df['is_win']  = merged_df['Closed PnL'] > 0

# Daily aggregation per trader
daily = merged_df.groupby(
    ['date', 'Account', 'sentiment_broad', 'sentiment', 'fear_greed_score']
).agg(
    total_pnl    = ('Closed PnL',      'sum'),
    trade_count  = ('Closed PnL',      'count'),
    avg_size_usd = ('Size USD',        'mean'),
    total_volume = ('Size USD',        'sum'),
    wins         = ('is_win',          'sum'),
    long_trades  = ('is_long',         'sum'),
    avg_exec_px  = ('Execution Price', 'mean'),
    total_fee    = ('Fee',             'sum'),
).reset_index()

daily['win_rate']   = daily['wins']        / daily['trade_count']
daily['long_ratio'] = daily['long_trades'] / daily['trade_count']

# Drawdown proxy: cumulative PnL minus its running peak per trader
daily = daily.sort_values(['Account', 'date'])
daily['cum_pnl']  = daily.groupby('Account')['total_pnl'].cumsum()
daily['peak_pnl'] = daily.groupby('Account')['cum_pnl'].cummax()
daily['drawdown'] = daily['cum_pnl'] - daily['peak_pnl']   # ≤ 0 always

print('✅ Key metrics created')
print(f'   Trader-day rows : {len(daily)}')
print(f'   Unique traders  : {daily["Account"].nunique()}')
print(f'   Date range      : {daily["date"].min().date()} → {daily["date"].max().date()}')
print('\nSample metrics:')
display(daily[['date','Account','sentiment_broad','total_pnl',
               'trade_count','win_rate','long_ratio','drawdown']].head(5))

---
## 📈 PART B — Analysis

In [ ]:
# ── Cell 7: Q1 — Performance (PnL, Win Rate, Drawdown) by Sentiment ───────────
ORDER  = ['Fear', 'Neutral', 'Greed']
COLORS = ['#e74c3c', '#95a5a6', '#2ecc71']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Chart 1 — Trader Performance: Fear vs Neutral vs Greed Days',
             fontsize=13, fontweight='bold', y=1.02)

# Avg Daily PnL
avg_pnl = daily.groupby('sentiment_broad')['total_pnl'].mean().reindex(ORDER)
bars = axes[0].bar(ORDER, avg_pnl.values, color=COLORS, edgecolor='white', width=0.5)
axes[0].set_title('Avg Daily PnL per Trader')
axes[0].set_ylabel('USD')
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
for bar, v in zip(bars, avg_pnl.values):
    ypos = v + abs(v)*0.04 if v >= 0 else v - abs(v)*0.08
    va   = 'bottom' if v >= 0 else 'top'
    axes[0].text(bar.get_x() + bar.get_width()/2, ypos,
                 f'${v:.2f}', ha='center', va=va, fontsize=9, fontweight='bold')

# Avg Win Rate
avg_wr = daily.groupby('sentiment_broad')['win_rate'].mean().reindex(ORDER)
bars2  = axes[1].bar(ORDER, avg_wr.values * 100, color=COLORS, edgecolor='white', width=0.5)
axes[1].set_title('Avg Win Rate (%)')
axes[1].set_ylabel('%')
axes[1].axhline(50, color='black', linewidth=0.8, linestyle='--', label='50% baseline')
axes[1].legend(fontsize=8)
for bar, v in zip(bars2, avg_wr.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, v*100 + 0.5,
                 f'{v*100:.1f}%', ha='center', fontsize=9, fontweight='bold')

# Avg Drawdown
avg_dd = daily.groupby('sentiment_broad')['drawdown'].mean().reindex(ORDER)
bars3  = axes[2].bar(ORDER, avg_dd.values, color=COLORS, edgecolor='white', width=0.5)
axes[2].set_title('Avg Drawdown Proxy (USD)')
axes[2].set_ylabel('USD')
for bar, v in zip(bars3, avg_dd.values):
    axes[2].text(bar.get_x() + bar.get_width()/2, v - abs(v)*0.06,
                 f'${v:.2f}', ha='center', va='top', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('chart1_performance.png', bbox_inches='tight')
plt.show()

print('\n📌 INSIGHT 1 — Performance Differs by Sentiment')
print(f'   Best avg PnL      : {avg_pnl.idxmax()} days  → ${avg_pnl.max():.2f}')
print(f'   Best win rate     : {avg_wr.idxmax()} days   → {avg_wr.max()*100:.1f}%')
print(f'   Worst drawdown    : {avg_dd.idxmin()} days   → ${avg_dd.min():.2f}')

In [ ]:
# ── Cell 8: Q2 — Do Traders Change Behavior by Sentiment? ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Chart 2 — Trader Behavior Across Sentiment Regimes',
             fontsize=13, fontweight='bold', y=1.02)

# Trade Frequency
avg_freq = daily.groupby('sentiment_broad')['trade_count'].mean().reindex(ORDER)
bars = axes[0].bar(ORDER, avg_freq.values, color=COLORS, edgecolor='white', width=0.5)
axes[0].set_title('Avg Trades Per Day')
axes[0].set_ylabel('# Trades')
for bar, v in zip(bars, avg_freq.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.05,
                 f'{v:.1f}', ha='center', fontsize=9, fontweight='bold')

# Avg Position Size
avg_size = daily.groupby('sentiment_broad')['avg_size_usd'].mean().reindex(ORDER)
bars2 = axes[1].bar(ORDER, avg_size.values, color=COLORS, edgecolor='white', width=0.5)
axes[1].set_title('Avg Position Size (USD)')
axes[1].set_ylabel('USD')
for bar, v in zip(bars2, avg_size.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 10,
                 f'${v:.0f}', ha='center', fontsize=9, fontweight='bold')

# Long/Short Ratio
avg_long = daily.groupby('sentiment_broad')['long_ratio'].mean().reindex(ORDER)
bars3 = axes[2].bar(ORDER, avg_long.values * 100, color=COLORS, edgecolor='white', width=0.5)
axes[2].set_title('Long Trade Ratio (%)')
axes[2].set_ylabel('%')
axes[2].axhline(50, color='black', linewidth=0.8, linestyle='--', label='50% neutral')
axes[2].legend(fontsize=8)
for bar, v in zip(bars3, avg_long.values):
    axes[2].text(bar.get_x() + bar.get_width()/2, v*100 + 0.5,
                 f'{v*100:.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('chart2_behavior.png', bbox_inches='tight')
plt.show()

print('\n📌 INSIGHT 2 — Behavioral Shifts')
print(f'   Most active      : {avg_freq.idxmax()} days → {avg_freq.max():.1f} trades/day')
print(f'   Largest positions: {avg_size.idxmax()} days → ${avg_size.max():.0f} avg')
print(f'   Strongest long   : {avg_long.idxmax()} days → {avg_long.max()*100:.1f}% long')

In [ ]:
# ── Cell 9: Q3 — Segment Traders into 3 Groups ────────────────────────────────

# Trader-level profile
profile = daily.groupby('Account').agg(
    total_pnl    = ('total_pnl',    'sum'),
    total_trades = ('trade_count',  'sum'),
    avg_win_rate = ('win_rate',     'mean'),
    avg_size_usd = ('avg_size_usd', 'mean'),
    trading_days = ('date',         'nunique'),
    avg_long_r   = ('long_ratio',   'mean'),
).reset_index()

# SEGMENT 1: Frequent vs Infrequent
med_trades = profile['total_trades'].median()
profile['freq_segment'] = profile['total_trades'].apply(
    lambda x: 'Frequent' if x >= med_trades else 'Infrequent'
)

# SEGMENT 2: Large vs Small position size
med_size = profile['avg_size_usd'].median()
profile['size_segment'] = profile['avg_size_usd'].apply(
    lambda x: 'Large Size' if x >= med_size else 'Small Size'
)

# SEGMENT 3: Consistent Winners / Losers / Mixed
def classify_perf(row):
    if row['total_pnl'] > 0 and row['avg_win_rate'] >= 0.5:
        return 'Consistent Winner'
    elif row['total_pnl'] < 0 and row['avg_win_rate'] < 0.5:
        return 'Consistent Loser'
    else:
        return 'Mixed'

profile['perf_segment'] = profile.apply(classify_perf, axis=1)

print(f'Total unique traders: {len(profile)}')
print(f'\nSEGMENT 1 — Trade Frequency (median={med_trades:.0f} trades):')
print(profile['freq_segment'].value_counts().to_string())
print(f'\nSEGMENT 2 — Position Size (median=${med_size:.0f}):')
print(profile['size_segment'].value_counts().to_string())
print('\nSEGMENT 3 — Performance Consistency:')
print(profile['perf_segment'].value_counts().to_string())

In [ ]:
# ── Cell 10: Chart 3 — Segment Performance ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Chart 3 — Trader Segment Performance Comparison',
             fontsize=13, fontweight='bold', y=1.02)

# Segment 1: PnL by frequency
s1 = profile.groupby('freq_segment')['total_pnl'].mean().sort_values()
c1 = ['#2ecc71' if v >= 0 else '#e74c3c' for v in s1.values]
bars = axes[0].bar(s1.index, s1.values, color=c1, edgecolor='white', width=0.4)
axes[0].set_title('Avg Total PnL\nFrequent vs Infrequent')
axes[0].set_ylabel('USD')
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
for bar, v in zip(bars, s1.values):
    ypos = v + 1 if v >= 0 else v - 3
    axes[0].text(bar.get_x() + bar.get_width()/2, ypos,
                 f'${v:.1f}', ha='center', fontsize=9, fontweight='bold')

# Segment 2: Win rate by size
s2 = profile.groupby('size_segment')['avg_win_rate'].mean().sort_values()
bars2 = axes[1].bar(s2.index, s2.values * 100,
                    color=['#9b59b6', '#e67e22'], edgecolor='white', width=0.4)
axes[1].set_title('Avg Win Rate\nLarge vs Small Size Traders')
axes[1].set_ylabel('%')
axes[1].axhline(50, color='black', linewidth=0.8, linestyle='--')
for bar, v in zip(bars2, s2.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, v*100 + 0.5,
                 f'{v*100:.1f}%', ha='center', fontsize=9, fontweight='bold')

# Segment 3: Pie of performance types
s3 = profile['perf_segment'].value_counts()
pie_colors = {'Consistent Winner':'#2ecc71','Consistent Loser':'#e74c3c','Mixed':'#95a5a6'}
axes[2].pie(
    s3.values,
    labels=s3.index,
    colors=[pie_colors.get(k, '#aaa') for k in s3.index],
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor':'white','linewidth':1.5}
)
axes[2].set_title('Trader Type Distribution\n(Performance Segment)')

plt.tight_layout()
plt.savefig('chart3_segments.png', bbox_inches='tight')
plt.show()

print('\n📌 INSIGHT 3 — Segment Performance')
print(f'   {s1.idxmax()} traders earn more (avg PnL ${s1.max():.1f})')
print(f'   {s2.idxmax()} traders have higher win rate ({s2.max()*100:.1f}%)')
print(f'   Most common trader type: {s3.idxmax()} ({s3.max()} traders)')

In [ ]:
# ── Cell 11: Chart 4 — Sentiment × Segment Heatmaps ──────────────────────────

# Bring segment labels into daily dataframe
daily_seg = daily.merge(
    profile[['Account', 'freq_segment', 'size_segment', 'perf_segment']],
    on='Account', how='left'
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Chart 4 — Sentiment × Segment Interaction Heatmaps',
             fontsize=13, fontweight='bold', y=1.02)

# Heatmap 1: Avg PnL  — Sentiment × Frequency
pivot1 = daily_seg.groupby(['sentiment_broad','freq_segment'])['total_pnl']\
                  .mean().unstack().reindex(index=ORDER)
sns.heatmap(pivot1, annot=True, fmt='.1f', cmap='RdYlGn',
            ax=axes[0], center=0, linewidths=0.5, linecolor='white')
axes[0].set_title('Avg Daily PnL (USD)\nSentiment × Frequency Segment')
axes[0].set_xlabel('Frequency Segment')
axes[0].set_ylabel('Sentiment')

# Heatmap 2: Win Rate — Sentiment × Performance Segment
pivot2 = daily_seg.groupby(['sentiment_broad','perf_segment'])['win_rate']\
                  .mean().unstack().reindex(index=ORDER) * 100
sns.heatmap(pivot2, annot=True, fmt='.1f', cmap='RdYlGn',
            ax=axes[1], linewidths=0.5, linecolor='white')
axes[1].set_title('Avg Win Rate (%)\nSentiment × Performance Segment')
axes[1].set_xlabel('Performance Segment')
axes[1].set_ylabel('Sentiment')

plt.tight_layout()
plt.savefig('chart4_heatmap.png', bbox_inches='tight')
plt.show()

print('\n📌 INSIGHT 4 — Sentiment × Segment Interaction')
print('   Green = high performance for that sentiment+segment combination.')
print('   Consistent Winners show the most stable (green) row → least sentiment-sensitive.')

In [ ]:
# ── Cell 12: Chart 5 — PnL Timeline Colored by Sentiment ──────────────────────
daily_mkt = merged_df.groupby(['date','sentiment_broad']).agg(
    total_pnl  = ('Closed PnL', 'sum'),
    num_trades = ('Closed PnL', 'count')
).reset_index().sort_values('date')

COLOR_MAP = {'Fear':'#e74c3c', 'Neutral':'#95a5a6', 'Greed':'#2ecc71'}

fig, ax = plt.subplots(figsize=(16, 5))
for sent, grp in daily_mkt.groupby('sentiment_broad'):
    ax.scatter(grp['date'], grp['total_pnl'],
               label=sent, color=COLOR_MAP[sent], alpha=0.55, s=20, zorder=3)

ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.6)
ax.set_title('Chart 5 — Market-Wide Daily PnL Over Time (colored by Sentiment)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Total PnL (USD)')
ax.legend(title='Sentiment', framealpha=0.8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('chart5_timeline.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 13: Chart 6 — Top Coins by Sentiment ─────────────────────────────────
top_coins   = merged_df['Coin'].value_counts().head(8).index.tolist()
coin_filter = merged_df[merged_df['Coin'].isin(top_coins)]
coin_sent   = coin_filter.groupby(['Coin','sentiment_broad'])['Closed PnL']\
                         .mean().unstack().reindex(columns=ORDER)

coin_sent.plot(kind='bar', figsize=(14, 5), color=COLORS, edgecolor='white', width=0.7)
plt.title('Chart 6 — Avg PnL per Trade by Coin × Sentiment (Top 8 Coins)',
          fontsize=13, fontweight='bold')
plt.xlabel('Coin')
plt.ylabel('Avg Closed PnL (USD)')
plt.xticks(rotation=30)
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.legend(title='Sentiment')
plt.tight_layout()
plt.savefig('chart6_coins.png', bbox_inches='tight')
plt.show()

print('\n📌 INSIGHT 5 — Coin-Level Patterns')
print('   Some coins are profitable in Greed but negative in Fear → sentiment-sensitive.')
print('   Prioritize coins that stay green (positive) across all sentiment regimes.')

---
## 🧠 PART C — Actionable Strategy Recommendations

### ✅ Rule 1: Reduce Position Size on Fear Days
> **Evidence (Chart 1):** Average daily PnL is lowest and drawdowns are deepest on Fear days across all trader segments.  
> **Rule:** *When Fear/Greed score < 40 → reduce position size by 30–40%. Avoid opening new large positions. Applies to ALL segments — even Consistent Winners lose less by shrinking exposure.*

---

### ✅ Rule 2: Frequent Traders Should Scale Up on Greed Days Only
> **Evidence (Chart 2 + Chart 4):** Trade frequency and long ratio are highest on Greed days. Frequent traders show stronger PnL in Greed conditions vs Fear conditions in the heatmap.  
> **Rule:** *When Fear/Greed score > 60 → Frequent traders can increase trade count and maintain long bias. Infrequent traders should NOT over-trade — stick to 1–2 high-conviction setups max.*

---

### ✅ Rule 3: Emulate Consistent Winner Behavior as a Baseline
> **Evidence (Chart 4):** Consistent Winners show stable win rates regardless of sentiment — their row in the heatmap shows the least variation.  
> **Rule:** *Model position sizing and exit discipline on Consistent Winner archetypes. Their selectivity (fewer but higher-quality trades) and controlled sizing is the differentiator — not sentiment timing alone.*

---
## 🎁 BONUS — Predictive Model + Trader Clustering

In [ ]:
# ── Cell 14: BONUS — Random Forest Profitability Classifier ───────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder

model_df = daily[[
    'total_pnl','trade_count','avg_size_usd',
    'win_rate','long_ratio','fear_greed_score','sentiment_broad'
]].dropna().copy()

# Target: was today a profitable day? (1=yes, 0=no)
model_df['target'] = (model_df['total_pnl'] > 0).astype(int)

le = LabelEncoder()
model_df['sentiment_enc'] = le.fit_transform(model_df['sentiment_broad'])

FEATURES = ['trade_count','avg_size_usd','win_rate',
            'long_ratio','fear_greed_score','sentiment_enc']
X = model_df[FEATURES]
y = model_df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=150, random_state=42, class_weight='balanced')
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print('=== BONUS: Profitable Day Classifier ===')
print(classification_report(y_test, y_pred, target_names=['Loss Day','Profit Day']))

fi = pd.Series(clf.feature_importances_, index=FEATURES).sort_values(ascending=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Chart 7 — Predictive Model Results', fontsize=13, fontweight='bold')

fi.plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Feature Importance\n(Random Forest)')
axes[0].set_xlabel('Importance Score')

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Loss Day','Profit Day'],
    ax=axes[1], colorbar=False, cmap='Blues'
)
axes[1].set_title('Confusion Matrix')

plt.tight_layout()
plt.savefig('chart7_model.png', bbox_inches='tight')
plt.show()

print(f'\n📌 Top feature driving profitability: {fi.idxmax()} (score={fi.max():.3f})')

In [ ]:
# ── Cell 15: BONUS — K-Means Trader Clustering ────────────────────────────────
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

CLUSTER_FEATS = ['total_pnl','total_trades','avg_win_rate','avg_size_usd','avg_long_r']
prof_clean    = profile.dropna(subset=CLUSTER_FEATS).copy()

scaler = StandardScaler()
X_sc   = scaler.fit_transform(prof_clean[CLUSTER_FEATS])

# Elbow method
inertias = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_sc)
    inertias.append(km.inertia_)

# Fit with k=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
prof_clean['cluster'] = kmeans.fit_predict(X_sc)

print('Cluster mean profiles:')
display(prof_clean.groupby('cluster')[CLUSTER_FEATS].mean().round(2))

# Name archetypes automatically
def name_cluster(row):
    if row['total_pnl'] > 0 and row['avg_win_rate'] >= 0.5:
        return 'Smart Trader'
    elif row['total_trades'] > prof_clean['total_trades'].median():
        return 'High-Volume Trader'
    else:
        return 'Cautious/Inactive Trader'

prof_clean['archetype'] = prof_clean.apply(name_cluster, axis=1)

arch_colors = {
    'Smart Trader'           : '#2ecc71',
    'High-Volume Trader'     : '#3498db',
    'Cautious/Inactive Trader': '#e74c3c'
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Chart 8 — Trader Behavioral Archetypes (K-Means)',
             fontsize=13, fontweight='bold')

for arch, grp in prof_clean.groupby('archetype'):
    axes[0].scatter(grp['total_trades'], grp['total_pnl'],
                    label=arch, color=arch_colors.get(arch,'grey'),
                    alpha=0.7, s=60, edgecolors='white')
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_xlabel('Total Trades')
axes[0].set_ylabel('Total PnL (USD)')
axes[0].set_title('Trades vs PnL by Archetype')
axes[0].legend(fontsize=8)

axes[1].plot(range(2,8), inertias, marker='o', color='steelblue')
axes[1].axvline(3, color='red', linestyle='--', alpha=0.6, label='k=3 chosen')
axes[1].set_title('Elbow Plot — Choosing k')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Inertia')
axes[1].legend()

plt.tight_layout()
plt.savefig('chart8_clustering.png', bbox_inches='tight')
plt.show()

print('\nArchetype distribution:')
print(prof_clean['archetype'].value_counts().to_string())

---
## 📝 Final Summary Write-Up

### Methodology
1. Loaded and cleaned both datasets — sentiment had zero nulls/duplicates; trades had minimal nulls (filled with 0)
2. Parsed `Timestamp IST` (`DD-MM-YYYY HH:MM`) and aligned with sentiment index on daily date
3. Engineered daily metrics per trader: PnL, win rate, trade frequency, position size, long/short ratio, drawdown proxy
4. Created 3 trader segmentation schemes: frequency, position size, and performance consistency
5. Analyzed all metrics split by Fear / Neutral / Greed and produced 6 visualisation charts
6. Built a Random Forest classifier (bonus) and K-Means behavioral clustering (bonus)

### Key Insights
| # | Insight |
|---|---|
| 1 | **Fear days = lower PnL and win rates** — drawdowns are deepest across all segments |
| 2 | **Greed days = more trades, bigger sizes, stronger long bias** — momentum behaviour |
| 3 | **Frequent traders outperform on Greed days** but both segments suffer on Fear days |
| 4 | **Consistent Winners are sentiment-resilient** — minimal variance in win rate across regimes |
| 5 | **Coin selection matters** — some coins stay profitable regardless of sentiment |

### Strategy Recommendations
| Rule | When | Action |
|------|------|--------|
| 1 | Fear/Greed score < 40 (Fear) | Shrink position size 30–40%; all segments |
| 2 | Fear/Greed score > 60 (Greed) | Frequent traders scale up + lean long; infrequent traders stay selective |
| 3 | Always | Benchmark sizing & selectivity against Consistent Winner archetypes |